In [14]:
# usar_modelo.py
# Carga el modelo entrenado y predice partidos por NOMBRE de equipo.
# Esto es lo que la app llamará.

import joblib
import numpy as np
import pandas as pd
from scipy.stats import poisson

# --- Cargar lo guardado (se hace una vez, al importar el módulo) ---
modelo = joblib.load("data/modelo_poisson.joblib")
elo_fifa = joblib.load("data/elo_fifa.joblib")

# --- Función auxiliar: la matriz de marcadores (la misma de 2.PepararDatos_ElegirModelo.ipynb)
def matriz_marcadores(lambda_local, lambda_visitante, max_goles=6):
    probs_local = [poisson.pmf(i, lambda_local) for i in range(max_goles + 1)]
    probs_visitante = [poisson.pmf(j, lambda_visitante) for j in range(max_goles + 1)]
    matriz = np.zeros((max_goles + 1, max_goles + 1))
    for i in range(max_goles + 1):
        for j in range(max_goles + 1):
            matriz[i, j] = probs_local[i] * probs_visitante[j]
    return matriz

# --- LA FUNCIÓN ESTRELLA: de nombres a predicción ---
def predecir_partido(equipo_local, equipo_visitante, es_neutral=True):
    # 1. Traducir nombres a Elo usando el diccionario.
    elo_local = elo_fifa[equipo_local]
    elo_visit = elo_fifa[equipo_visitante]

    # 2. Armar las variables para CADA equipo, en el orden que el modelo espera:
    #    [mi_elo, elo_rival, es_local]
    #    Para el local: su elo, el del rival, y es_local=1 si NO es neutral.
    es_local_flag = 0 if es_neutral else 1
    X_local = pd.DataFrame([[elo_local, elo_visit, es_local_flag]],
                           columns=["mi_elo", "elo_rival", "es_local"])
    # Para el visitante: su elo, el del local, y es_local=0 (nunca juega en casa).
    X_visit = pd.DataFrame([[elo_visit, elo_local, 0]],
                           columns=["mi_elo", "elo_rival", "es_local"])

    # 3. Predecir el λ de cada equipo con el modelo.
    lambda_local = modelo.predict(X_local)[0]
    lambda_visit = modelo.predict(X_visit)[0]

    # 4. Construir la matriz y sacar los resultados (como antes).
    m = matriz_marcadores(lambda_local, lambda_visit)
    idx = np.unravel_index(np.argmax(m), m.shape)

    prob_gana_local = prob_empate = prob_gana_visit = 0.0
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            if i > j:   prob_gana_local += m[i, j]
            elif i == j: prob_empate += m[i, j]
            else:        prob_gana_visit += m[i, j]

    return {
        "equipos": f"{equipo_local} vs {equipo_visitante}",
        "lambda_local": round(lambda_local, 2),
        "lambda_visitante": round(lambda_visit, 2),
        "marcador_probable": (int(idx[0]), int(idx[1])),
        "prob_marcador": round(m[idx], 3),
        "gana_local": round(prob_gana_local, 3),
        "empate": round(prob_empate, 3),
        "gana_visitante": round(prob_gana_visit, 3),
    }

# --- Prueba ---
if __name__ == "__main__":
    resultado = predecir_partido("Argentina", "Cape Verde", es_neutral=True)
    for k, v in resultado.items():
        print(f"{k}: {v}")

equipos: Argentina vs Cape Verde
lambda_local: 2.93
lambda_visitante: 0.47
marcador_probable: (2, 0)
prob_marcador: 0.144
gana_local: 0.837
empate: 0.098
gana_visitante: 0.036
